In [1]:
from __future__ import annotations

import urllib.request
from pathlib import Path

from ultralytics import YOLO

# Notebook assumed run with cwd = Week6 (same folder as this file).
ROOT = Path.cwd().resolve()
DATA_YAML = ROOT / "aquarium" / "data.yaml"
assert DATA_YAML.is_file(), f"Missing {DATA_YAML} — set kernel cwd to Week6."

PRETRAINED_NAME = "yolo26n.pt"
PRETRAINED = ROOT / PRETRAINED_NAME
if not PRETRAINED.is_file():
    url = "https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26n.pt"
    print("Downloading", url)
    urllib.request.urlretrieve(url, PRETRAINED)

RUNS = ROOT / "runs"
FT_NAME = "aquarium_ft"
BEST_PT = RUNS / FT_NAME / "weights" / "best.pt"


In [2]:
def metrics_dict(m) -> dict:
    """Ultralytics DetMetrics → scalar summary."""
    box = m.box
    return {
        "precision": float(box.mp),
        "recall": float(box.mr),
        "mAP50": float(box.map50),
        "mAP50-95": float(box.map),
    }


def print_metrics(title: str, d: dict) -> None:
    print(title)
    print(f"  Precision: {d['precision']:.4f}")
    print(f"  Recall:    {d['recall']:.4f}")
    print(f"  mAP50:     {d['mAP50']:.4f}")
    print(f"  mAP50-95:  {d['mAP50-95']:.4f}")


In [3]:
model_pretrained = YOLO(str(PRETRAINED))
metrics_pre = model_pretrained.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    plots=True,
    project=str(RUNS),
    name="val_pretrained_test",
)
pretrained_results = metrics_dict(metrics_pre)
print_metrics("Pretrained (yolo26n.pt) on test set", pretrained_results)


Ultralytics 8.4.31  Python-3.11.9 torch-2.6.0+cpu CPU (13th Gen Intel Core(TM) i9-13900H)
YOLO26n summary (fused): 122 layers, 2,408,932 parameters, 0 gradients, 5.4 GFLOPs
val: Fast image access  (ping: 0.30.2 ms, read: 181.261.8 MB/s, size: 105.5 KB)
val: Scanning C:\Users\nguye\OneDrive\Máy tính\University courses\Sem1_2026\COS30082\Week6\aquarium\test\labels.cache... 63 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 63/63  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.1it/s 3.7s1.4ss
                   all         63        584     0.0279     0.0545      0.025     0.0185
                person         30        249     0.0223      0.145     0.0138    0.00901
               bicycle         11        154          0          0          0          0
                   car          7         82          0          0          0          0
            motorcycle          6         35          0          0       

In [4]:
model_ft = YOLO(str(PRETRAINED))
model_ft.train(
    data=str(DATA_YAML),
    epochs=15,
    imgsz=640,
    freeze=10,
    project=str(RUNS),
    name=FT_NAME,
)

assert BEST_PT.is_file(), f"Expected weights at {BEST_PT}"
print("Best weights:", BEST_PT)


New https://pypi.org/project/ultralytics/8.4.38 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.31  Python-3.11.9 torch-2.6.0+cpu CPU (13th Gen Intel Core(TM) i9-13900H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\nguye\OneDrive\My tnh\University courses\Sem1_2026\COS30082\Week6\aquarium\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\Users

COMET ERROR: Failed to create Comet experiment, reason: ValueError('Comet.ml requires an API key. Please provide as the first argument to Experiment(api_key) or as an environment variable named COMET_API_KEY ')


WARNING Comet installed but not initialized correctly, not logging this run. Comet.ml requires an API key. Please provide as the first argument to Experiment(api_key) or as an environment variable named COMET_API_KEY 
Overriding model.yaml nc=80 with nc=7

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.module

In [5]:
model_best = YOLO(str(BEST_PT))
metrics_ft = model_best.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    plots=True,
    project=str(RUNS),
    name="val_finetuned_test",
)
finetuned_results = metrics_dict(metrics_ft)
print_metrics("Fine-tuned (best.pt) on test set", finetuned_results)


Ultralytics 8.4.31  Python-3.11.9 torch-2.6.0+cpu CPU (13th Gen Intel Core(TM) i9-13900H)
YOLO26n summary (fused): 122 layers, 2,376,201 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1239.1368.3 MB/s, size: 95.7 KB)
val: Scanning C:\Users\nguye\OneDrive\Máy tính\University courses\Sem1_2026\COS30082\Week6\aquarium\test\labels.cache... 63 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 63/63  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.1it/s 3.6s1.4ss
                   all         63        584      0.651      0.465      0.553      0.328
                  fish         30        249       0.61      0.386      0.499      0.278
             jellyfish         11        154      0.618      0.708      0.733      0.425
               penguin          7         82      0.589      0.451      0.516      0.216
                puffin          6         35      0.589      0.343      

In [7]:
rows = [
    ("Pretrained yolo26n (COCO)", pretrained_results),
    ("Fine-tuned best.pt", finetuned_results),
]
print("Stage | Precision | Recall | mAP50 | mAP50-95")
for label, d in rows:
    print(
        f"{label} | {d['precision']:.4f} | {d['recall']:.4f} | {d['mAP50']:.4f} | {d['mAP50-95']:.4f}"
    )

report_path = ROOT / "lab06_report.md"
lines = [
    "# Lab 06 — YOLO aquarium results", 
    "## Test-set metrics (Ultralytics `model.val`, `split=test`)",
    "Stage | Precision | Recall | mAP50 | mAP50-95",
]
for label, d in rows:
    lines.append(
        f"{label} | {d['precision']:.4f} | {d['recall']:.4f} | {d['mAP50']:.4f} | {d['mAP50-95']:.4f}"
    )
lines.append("Training outputs under `runs_lab06/`.")
report_path.write_text("\n".join(lines), encoding="utf-8")
print("Wrote", report_path)


Stage | Precision | Recall | mAP50 | mAP50-95
Pretrained yolo26n (COCO) | 0.0279 | 0.0545 | 0.0250 | 0.0185
Fine-tuned best.pt | 0.6507 | 0.4651 | 0.5534 | 0.3278
Wrote C:\Users\nguye\OneDrive\Máy tính\University courses\Sem1_2026\COS30082\Week6\lab06_report.md
